In [ ]:
# Summary of Analysis Results
print("\n" + "="*60)
print("WASSERSTEIN RISK INDEX ANALYSIS SUMMARY")
print("="*60)
print(f"\nData Shape: {df_returns.shape}")
print(f"Number of Assets: {len(df_returns.columns)}")
print(f"\nPortfolio Risk Metrics:")
for key, value in wasserstein_risk_index.items():
    print(f"  {key}: {value:.6f}")
print(f"\nWasserstein Distance (W{config.DEFAULT_WASSERSTEIN_P}): {w_distance:.6f}")
print(f"Mean Asset Volatility: {risk_metrics_df['std'].mean():.6f}")
print("\nAnalysis complete! Check the output/ directory for saved results.")
print("="*60)

In [ ]:
# Save results to CSV files
output_dict = {
    'risk_metrics.csv': risk_metrics_df,
    'wasserstein_distances.csv': w_distances_df,
    'portfolio_returns_sample.csv': pd.DataFrame({'Portfolio_Returns': portfolio_returns[:100]})
}

for filename, dataframe in output_dict.items():
    filepath = f'{config.OUTPUT_PATH}{filename}'
    try:
        dataframe.to_csv(filepath)
        print(f"✓ Saved: {filepath}")
    except FileNotFoundError:
        print(f"⚠ Output directory not found: {config.OUTPUT_PATH}")
        print(f"  Preview of {filename}:")
        print(dataframe.head())

## 8. Export Analysis

In [ ]:
# Visualize portfolio returns distribution with risk metrics
portfolio_returns = df_returns.mean(axis=1).values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(portfolio_returns, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(portfolio_returns.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {portfolio_returns.mean():.4f}')
axes[0].axvline(portfolio_returns.mean() - portfolio_returns.std(), color='orange', linestyle='--', linewidth=2, label=f'±1σ: {portfolio_returns.std():.4f}')
axes[0].set_title('Portfolio Returns Distribution', fontsize=12)
axes[0].set_xlabel('Returns')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Cumulative distribution
sorted_returns = np.sort(portfolio_returns)
cumulative = np.arange(1, len(sorted_returns) + 1) / len(sorted_returns)
axes[1].plot(sorted_returns, cumulative, linewidth=2, color='steelblue')
axes[1].set_title('Cumulative Distribution of Portfolio Returns', fontsize=12)
axes[1].set_xlabel('Returns')
axes[1].set_ylabel('Cumulative Probability')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize risk metrics by asset
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = ['mean', 'std', 'min', 'max']
for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    values = risk_metrics_df[metric]
    colors = plt.cm.viridis(np.linspace(0, 1, len(values)))
    ax.bar(values.index, values.values, color=colors, edgecolor='black')
    ax.set_title(f'{metric.upper()} Returns by Asset', fontsize=12)
    ax.set_ylabel(metric.capitalize())
    ax.set_xlabel('Assets')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize pairwise Wasserstein distances as heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(w_distances_df, annot=True, fmt='.4f', cmap='YlOrRd', cbar_kws={'label': 'Wasserstein Distance'})
plt.title('Pairwise Wasserstein Distances Between Assets', fontsize=14)
plt.xlabel('Assets')
plt.ylabel('Assets')
plt.tight_layout()
plt.show()

## 7. Visualize Results

In [ ]:
# Compute risk metrics for each asset
risk_metrics = {}
for col in df_returns.columns:
    metrics = analysis.compute_risk_metrics(df_returns[col].values)
    risk_metrics[col] = metrics

risk_metrics_df = pd.DataFrame(risk_metrics).T
print("Risk Metrics by Asset:")
print(risk_metrics_df)

# Compute Wasserstein Risk Index as weighted combination of metrics
portfolio_returns = df_returns.mean(axis=1).values
wasserstein_risk_index = analysis.compute_risk_metrics(portfolio_returns)

print("\nPortfolio Wasserstein Risk Index:")
for key, value in wasserstein_risk_index.items():
    print(f"  {key}: {value:.6f}")

## 6. Compute Risk Index

In [ ]:
# Split data into two distributions for comparison
split_idx = len(df_returns) // 2
X_dist1 = df_returns.iloc[:split_idx].values
X_dist2 = df_returns.iloc[split_idx:].values

# Compute Wasserstein distance
w_distance = analysis.wasserstein_distance(X_dist1, X_dist2, p=config.DEFAULT_WASSERSTEIN_P)

print(f"Wasserstein Distance (W{config.DEFAULT_WASSERSTEIN_P}):")
print(f"  Between two data distributions: {w_distance:.6f}")

# Compute pairwise Wasserstein distances between assets
asset_names = df_returns.columns
n_assets = len(asset_names)
w_distances = np.zeros((n_assets, n_assets))

for i in range(n_assets):
    for j in range(n_assets):
        if i != j:
            X_i = df_returns.iloc[:, i].values.reshape(-1, 1)
            X_j = df_returns.iloc[:, j].values.reshape(-1, 1)
            w_distances[i, j] = analysis.wasserstein_distance(X_i, X_j)

w_distances_df = pd.DataFrame(w_distances, index=asset_names, columns=asset_names)
print("\nPairwise Wasserstein Distances:")
print(w_distances_df)

## 5. Calculate Wasserstein Distance

In [ ]:
# Normalize the features
X_normalized = features.normalize_features(df_returns)
X_normalized_df = pd.DataFrame(
    X_normalized,
    columns=[f'{col}_norm' for col in df_returns.columns]
)

print("Normalized Features Statistics:")
print(X_normalized_df.describe())

# Scale features to [0, 1] range
X_scaled = features.scale_features(df_returns)
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=[f'{col}_scaled' for col in df_returns.columns]
)

print("\nScaled Features Statistics:")
print(X_scaled_df.describe())

## 4. Feature Engineering

In [ ]:
# Visualize data distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(df_returns.columns):
    axes[i].hist(df_returns[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution: {col}')
    axes[i].set_xlabel('Returns')
    axes[i].set_ylabel('Frequency')

fig.suptitle('Asset Return Distributions', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Generate synthetic portfolio data for demonstration
n_samples = 1000
n_assets = 5

# Create synthetic returns data
synthetic_returns = np.random.multivariate_normal(
    mean=np.random.uniform(0.001, 0.005, n_assets),
    cov=np.eye(n_assets) * 0.01,
    size=n_samples
)

# Create DataFrame
df_returns = pd.DataFrame(
    synthetic_returns,
    columns=[f'Asset_{i+1}' for i in range(n_assets)]
)

print("Data Summary Statistics:")
print(df_returns.describe())
print("\nData Shape:", df_returns.shape)

## 3. Load and Explore Data

In [ ]:
print("Configuration Settings:")
print(f"Data Path: {config.DATA_PATH}")
print(f"Output Path: {config.OUTPUT_PATH}")
print(f"Random Seed: {config.RANDOM_SEED}")
print(f"Test Size: {config.TEST_SIZE}")
print(f"Default Wasserstein P: {config.DEFAULT_WASSERSTEIN_P}")

# Set random seed for reproducibility
np.random.seed(config.RANDOM_SEED)

## 2. Load Configuration Settings

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import from algogators_wrisk package
from algogators_wrisk import config
from algogators_wrisk import data
from algogators_wrisk import features
from algogators_wrisk import analysis

# Set visualization style
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Import Required Libraries

# Wasserstein Risk Index Analysis

This notebook implements the Wasserstein Risk Index for portfolio and market risk analysis using optimal transport theory.
